### Imports and Loading Model

In [1]:
# Import
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
import chromadb

C:\Users\Raghu Sanivarapu\venvs\simple-rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3243.49it/s]

### Knowledge Base

In [3]:
# 16 IT help desk articles, each one starting with a "## " heading. Some share a word with a different meaning
# (lock, reset, port, team), and some hold near-identical facts (two error codes, two printer IPs, an old VPN)
knowledge_base = """
## VPN Timeout

If the VPN times out when you work from home, check your internet connection and restart the VPN app.
The VPN needs UDP ports 500 and 4500 open. If it still fails, contact the network team.

## Old VPN (Retired)

The old Cisco VPN used TCP port 1723. It was retired in 2024 and no longer works.

## USB Port Not Working

If a USB port stops working, try another port, then restart the laptop.
Docking station ports work only when the dock's own power cable is plugged in.

## MFA Setup

If Microsoft Authenticator won't finish setting up MFA, first set your phone's date and time to automatic.
Then reinstall the app and scan the QR code again.

## Account Locked

After 5 wrong passwords your account locks for 15 minutes. Wait, or call the help desk to unlock it.

## Laptop Cable Lock

Every laptop on an open desk must be fastened with a cable lock. Facilities keeps spare lock keys.

## Screen Lock

Your screen locks itself after 10 minutes without use. Press Windows+L to lock it when you leave your desk.

## Password Reset

A new password needs at least 12 characters, with an uppercase letter, a lowercase letter, a number and a special character.
If the reset page shows error 0x80070005, try another browser, or call the help desk for a manual reset.

## Phone Factory Reset

Before returning a company phone, remove the work profile, then factory reset it in Settings.

## Disk Error

Error 0x80070057 during a Windows update means the disk is nearly full. Run Disk Cleanup and try the update again.

## Slow Laptop

If your laptop is slow, let Windows updates finish, close unused browser tabs and restart.
Keep at least 15% of the disk free.

## Office WiFi

Connect to the CorpWiFi-5G network. If the WiFi keeps dropping, move closer to an access point
or restart your laptop's WiFi. Building C has older access points, due for an upgrade.

## Floor 3 Printer

To add the HP-Floor3-Color printer, connect to the office network (not guest WiFi) and add the printer
by its IP address, 10.50.3.25.

## Floor 2 Printer

The HP-Floor2-Mono printer prints in black and white only. Its IP address is 10.50.2.18.

## Teams Audio

If you hear nothing in Teams meetings, pick the right speaker in Teams settings, make sure Windows isn't muted,
and restart Teams.

## New Team Member

When someone new joins your team, your manager orders their laptop and accounts two weeks before they start.
"""

# One article per heading: the heading becomes its "title", the rest its text
articles = MarkdownHeaderTextSplitter([("##", "title")]).split_text(knowledge_base)
print("Articles:", len(articles))

Articles: 16


### Chunking

In [4]:
# Cut each article into chunks of up to 150 characters: a longer article splits at its line breaks, a shorter one stays whole
splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=0)
chunks = splitter.split_documents(articles)   # each chunk keeps its article's title

print("Chunks:", len(chunks))
for i, chunk in enumerate(chunks):
    print(f"C{i+1} [{chunk.metadata['title']}] {chunk.page_content}")

Chunks: 20
C1 [VPN Timeout] If the VPN times out when you work from home, check your internet connection and restart the VPN app.
C2 [VPN Timeout] The VPN needs UDP ports 500 and 4500 open. If it still fails, contact the network team.
C3 [Old VPN (Retired)] The old Cisco VPN used TCP port 1723. It was retired in 2024 and no longer works.
C4 [USB Port Not Working] If a USB port stops working, try another port, then restart the laptop.
Docking station ports work only when the dock's own power cable is plugged in.
C5 [MFA Setup] If Microsoft Authenticator won't finish setting up MFA, first set your phone's date and time to automatic.
C6 [MFA Setup] Then reinstall the app and scan the QR code again.
C7 [Account Locked] After 5 wrong passwords your account locks for 15 minutes. Wait, or call the help desk to unlock it.
C8 [Laptop Cable Lock] Every laptop on an open desk must be fastened with a cable lock. Facilities keeps spare lock keys.
C9 [Screen Lock] Your screen locks itself after 10 m

### Indexing Data

In [5]:
texts = [chunk.page_content for chunk in chunks]
titles = [chunk.metadata for chunk in chunks]
ids = [f"C{i+1}" for i in range(len(chunks))]

# Setup Chroma client and collection, in memory
chroma_client = chromadb.Client()
# Drop existing collection if it already exists
try:
    chroma_client.delete_collection("it_kb")
except Exception:
    pass

# Now create fresh: "cosine" makes the distance 1 - cosine similarity
collection = chroma_client.create_collection("it_kb", configuration={"hnsw": {"space": "cosine"}})

# Add chunks with embeddings
embeddings = model.encode(texts).tolist()
collection.add(ids=ids, documents=texts, embeddings=embeddings, metadatas=titles)

print("Indexed chunks:", collection.count())

Indexed chunks: 20


### Basic RAG Query

In [6]:
# Each question, then the article it should come from. Basic search ranks by closeness of meaning only,
# so a chunk that shares a word with the question ("lock", "reset", "port") can beat the right one
questions = {
    "How do I unlock my screen?": "Screen Lock",
    "I lost the key for my lock": "Laptop Cable Lock",
    "How do I reset my password?": "Password Reset",
    "Which port does the VPN use?": "VPN Timeout",
    "What does error 0x80070057 mean?": "Disk Error",
}

for query, expected in questions.items():
    query_emb = model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_emb, n_results=3)

    top_title = results["metadatas"][0][0]["title"]
    verdict = "RIGHT" if top_title == expected else "WRONG"
    print()
    print(f"Question: {query}")
    print(f"Should come from: {expected}  ->  top result is {verdict}")
    for chunk_id, text, meta, distance in zip(results["ids"][0], results["documents"][0], results["metadatas"][0], results["distances"][0]):
        print(f"  {chunk_id} [{meta['title']}] similarity={1 - distance:.4f}: {text}")


Question: How do I unlock my screen?
Should come from: Screen Lock  ->  top result is RIGHT
  C9 [Screen Lock] similarity=0.6488: Your screen locks itself after 10 minutes without use. Press Windows+L to lock it when you leave your desk.
  C7 [Account Locked] similarity=0.4231: After 5 wrong passwords your account locks for 15 minutes. Wait, or call the help desk to unlock it.
  C12 [Phone Factory Reset] similarity=0.3496: Before returning a company phone, remove the work profile, then factory reset it in Settings.

Question: I lost the key for my lock
Should come from: Laptop Cable Lock  ->  top result is WRONG
  C7 [Account Locked] similarity=0.4472: After 5 wrong passwords your account locks for 15 minutes. Wait, or call the help desk to unlock it.
  C9 [Screen Lock] similarity=0.3566: Your screen locks itself after 10 minutes without use. Press Windows+L to lock it when you leave your desk.
  C8 [Laptop Cable Lock] similarity=0.3242: Every laptop on an open desk must be fastened w